# Sales Dashboard for Regional Performance

SQL, Power BI, Python  
Dataset: 9,994 rows, 4 regions, 3 product categories

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import sqlite3
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.titlesize': 12})

## 2. Load Data

In [ ]:
orders    = pd.read_csv('data/orders.csv', parse_dates=['order_date', 'ship_date'])
customers = pd.read_csv('data/customers.csv')
products  = pd.read_csv('data/products.csv')

print('orders   :', orders.shape)
print('customers:', customers.shape)
print('products :', products.shape)
print('date range:', orders['order_date'].min().date(), 'to', orders['order_date'].max().date())

In [ ]:
orders.head()

In [ ]:
print(orders.isnull().sum())
print()
print(orders.dtypes)

## 3. Feature Engineering

In [ ]:
orders['year']          = orders['order_date'].dt.year
orders['month']         = orders['order_date'].dt.month
orders['month_label']   = orders['order_date'].dt.strftime('%b %Y')
orders['quarter']       = orders['order_date'].dt.quarter
orders['days_to_ship']  = (orders['ship_date'] - orders['order_date']).dt.days
orders['profit_margin'] = orders['profit'] / orders['sales'].replace(0, np.nan)
orders['is_loss']       = orders['profit'] < 0

df = orders.merge(customers[['customer_id', 'segment', 'city', 'state']], on='customer_id', how='left')
print(df[['order_date', 'month_label', 'days_to_ship', 'profit_margin', 'is_loss']].head())

## 4. KPI Summary

In [ ]:
total_sales   = orders['sales'].sum()
total_profit  = orders['profit'].sum()
margin_pct    = total_profit / total_sales * 100
total_orders  = orders['order_id'].nunique()
avg_order_val = total_sales / total_orders
units_sold    = orders['quantity'].sum()
loss_orders   = orders['is_loss'].sum()
return_rate   = loss_orders / len(orders) * 100
avg_ship_days = orders['days_to_ship'].mean()

kpis = {
    'Total Sales':        f'${total_sales:,.2f}',
    'Total Profit':       f'${total_profit:,.2f}',
    'Profit Margin':      f'{margin_pct:.2f}%',
    'Total Orders':       f'{total_orders:,}',
    'Avg Order Value':    f'${avg_order_val:,.2f}',
    'Units Sold':         f'{units_sold:,}',
    'Loss Orders':        f'{loss_orders:,} ({return_rate:.1f}%)',
    'Avg Days to Ship':   f'{avg_ship_days:.1f}',
    'Regions':            str(orders['region'].nunique()),
    'Post-Festival Drop': '~14% South (see Section 6)',
}

for k, v in kpis.items():
    print(f'{k:<25} {v}')

## 5. Regional Sales Analysis

In [ ]:
region_stats = orders.groupby('region').agg(
    total_sales  = ('sales',    'sum'),
    total_profit = ('profit',   'sum'),
    total_orders = ('order_id', 'nunique'),
    total_units  = ('quantity', 'sum'),
    avg_discount = ('discount', 'mean')
).reset_index()
region_stats['margin_pct'] = region_stats['total_profit'] / region_stats['total_sales'] * 100
region_stats = region_stats.sort_values('total_sales', ascending=False)
print(region_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(region_stats['region'], region_stats['total_sales'] / 1000)
axes[0].set_title('Total Sales by Region')
axes[0].set_ylabel('Sales (K)')

axes[1].bar(region_stats['region'], region_stats['margin_pct'])
axes[1].set_title('Profit Margin % by Region')
axes[1].set_ylabel('Margin %')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

axes[2].bar(region_stats['region'], region_stats['total_orders'])
axes[2].set_title('Total Orders by Region')
axes[2].set_ylabel('Orders')

plt.tight_layout()
plt.savefig('regional_overview.png', bbox_inches='tight')
plt.show()

## 6. Monthly Sales Trend

In [ ]:
monthly = orders.groupby(['year', 'month', 'region'])['sales'].sum().reset_index()
monthly['period'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))
monthly = monthly.sort_values('period')

fig, ax = plt.subplots(figsize=(13, 5))
for region, grp in monthly.groupby('region'):
    ax.plot(grp['period'], grp['sales'] / 1000, marker='o', markersize=3, label=region)

ax.set_title('Monthly Sales by Region')
ax.set_xlabel('Month')
ax.set_ylabel('Sales (K)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.0f}K'))
ax.legend(title='Region')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('monthly_trend.png', bbox_inches='tight')
plt.show()

## 7. Post-Festival Sales Drop — South Region

In [ ]:
south = orders[orders['region'] == 'South'].copy()
festival_sales      = south[south['month'].isin([10, 11])]['sales'].sum()
post_festival_sales = south[south['month'].isin([12, 1])]['sales'].sum()
drop_pct = (post_festival_sales - festival_sales) / festival_sales * 100

print(f'Festival (Oct-Nov) Sales : ${festival_sales:,.2f}')
print(f'Post-Festival (Dec-Jan)  : ${post_festival_sales:,.2f}')
print(f'Change                   : {drop_pct:+.2f}%')

In [ ]:
rows = []
for region in orders['region'].unique():
    rdf = orders[orders['region'] == region]
    f  = rdf[rdf['month'].isin([10, 11])]['sales'].sum()
    pf = rdf[rdf['month'].isin([12, 1])]['sales'].sum()
    rows.append({'region': region, 'festival': f, 'post_festival': pf,
                 'pct_change': (pf - f) / f * 100})

rf = pd.DataFrame(rows).sort_values('pct_change')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rf['region'], rf['pct_change'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Post-Festival Sales Change by Region')
ax.set_ylabel('% Change')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for i, row in rf.reset_index().iterrows():
    ax.text(i, row['pct_change'] - 0.5, f"{row['pct_change']:.1f}%",
            ha='center', va='top', fontsize=10)
plt.tight_layout()
plt.savefig('festival_drop.png', bbox_inches='tight')
plt.show()

## 8. Product Category Analysis

In [ ]:
import os
if os.path.exists('data/dataset_original.csv'):
    orig = pd.read_csv('data/dataset_original.csv', encoding='latin1')
else:
    orig = pd.read_csv('/mnt/user-data/uploads/dataset.csv', encoding='latin1')
orig.columns = [c.lower().replace(' ', '_') for c in orig.columns]
orig['order_date'] = pd.to_datetime(orig['order_date'])

cat = orig.groupby(['category', 'sub_category']).agg(
    total_sales  = ('sales',   'sum'),
    total_profit = ('profit',  'sum'),
    total_qty    = ('quantity','sum')
).reset_index()
cat['margin_pct'] = cat['total_profit'] / cat['total_sales'] * 100
cat = cat.sort_values('total_sales', ascending=False)

top10 = cat.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10['sub_category'], top10['total_sales'] / 1000)
ax.set_xlabel('Total Sales (K)')
ax.set_title('Top 10 Sub-Categories by Sales')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.0f}K'))
plt.tight_layout()
plt.savefig('subcategory_sales.png', bbox_inches='tight')
plt.show()

## 9. Discount vs Profit

In [ ]:
orig['discount_band'] = pd.cut(orig['discount'],
    bins=[-0.01, 0.001, 0.10, 0.20, 0.30, 1.0],
    labels=['0%', '1-10%', '11-20%', '21-30%', '30%+'])

disc = orig.groupby('discount_band').agg(
    orders      = ('row_id', 'count'),
    avg_profit  = ('profit', 'mean'),
    loss_orders = ('profit', lambda x: (x < 0).sum())
).reset_index()
disc['loss_pct'] = disc['loss_orders'] / disc['orders'] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(disc['discount_band'], disc['avg_profit'])
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_title('Avg Profit by Discount Band')
ax1.set_ylabel('Avg Profit ($)')

ax2.bar(disc['discount_band'], disc['loss_pct'])
ax2.set_title('Loss Order Rate by Discount Band')
ax2.set_ylabel('Loss %')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('discount_analysis.png', bbox_inches='tight')
plt.show()

print('Loss rate at 30%+ discount:',
      round(disc[disc['discount_band'] == '30%+']['loss_pct'].values[0], 1), '%')

## 10. Customer Segment by Region

In [ ]:
seg = orig.groupby(['segment', 'region']).agg(
    total_sales = ('sales', 'sum')
).reset_index()

pivot = seg.pivot(index='segment', columns='region', values='total_sales').fillna(0)
ax = pivot.plot(kind='bar', figsize=(9, 4))
ax.set_title('Sales by Segment and Region')
ax.set_ylabel('Sales ($)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(title='Region')
plt.tight_layout()
plt.savefig('segment_region.png', bbox_inches='tight')
plt.show()

## 11. Ship Mode Efficiency

In [ ]:
orig['days_to_ship'] = (pd.to_datetime(orig['ship_date']) - pd.to_datetime(orig['order_date'])).dt.days

ship = orig.groupby('ship_mode').agg(
    avg_days = ('days_to_ship', 'mean'),
    min_days = ('days_to_ship', 'min'),
    max_days = ('days_to_ship', 'max'),
    orders   = ('order_id',     'nunique')
).reset_index().sort_values('avg_days')

print(ship.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ship['ship_mode'], ship['avg_days'])
ax.set_xlabel('Avg Days to Ship')
ax.set_title('Shipping Efficiency by Ship Mode')
for i, v in enumerate(ship['avg_days']):
    ax.text(v + 0.05, i, f'{v:.1f}', va='center')
plt.tight_layout()
plt.savefig('ship_mode.png', bbox_inches='tight')
plt.show()

## 12. SQL via sqlite3

In [ ]:
conn = sqlite3.connect(':memory:')
orig.to_sql('sales', conn, if_exists='replace', index=False)

result = pd.read_sql('''
    SELECT
        region,
        COUNT(DISTINCT order_id)              AS total_orders,
        ROUND(SUM(sales), 2)                  AS total_sales,
        ROUND(SUM(profit), 2)                 AS total_profit,
        ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margin_pct
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
''', conn)

print(result.to_string(index=False))
conn.close()

## 13. Export for Power BI

In [ ]:
export_cols = [
    'row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
    'customer_id', 'customer_name', 'segment', 'city', 'state', 'region',
    'product_id', 'product_name', 'category', 'sub_category',
    'sales', 'quantity', 'discount', 'profit'
]
export_df = orig[[c for c in export_cols if c in orig.columns]].copy()
export_df.to_csv('data/sales_analysis_ready.csv', index=False)
print(f'Exported {len(export_df):,} rows to data/sales_analysis_ready.csv')

## 14. Findings Summary

| Finding | Region | Note |
|---|---|---|
| West leads in total sales | West | highest revenue |
| East has highest order count | East | 2,848 orders |
| Post-festival sales drop 14% | South | recovery plan triggered |
| 30%+ discounts cause 50%+ loss rate | All | review discount policy |
| Tables sub-category has negative profit | All | pricing review needed |
| Phones and Chairs are top earners | All | priority stock items |
